In [39]:
import math 
import numpy as np 
import random

In [44]:
# The Value class 
class Value: 
    def __init__(self, data, _children=(), _op='', label=""):
        self.data = data 
        self._backward = lambda: None # This will store the function that does that chain rule at every node 
        self.grad = 0.0 
        self._prev = set(_children)
        self._op = _op 
        self.label = label 
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) # Lets us add constants to a value object
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # implementing the chain rule for gradient calculation over addition 
            # The local derivative for addition is always 1 
            # The upstream derivative in our logic is out.grad 
            self.grad += 1.0 * out.grad 
            other.grad += 1.0 * out.grad 

        out._backward = _backward 
        return out 

    def __mul__(self, other): 
        other = other if isinstance(other, Value) else Value(other) # Lets us add constants to a value object 
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # for a gradient calculation over multiplication, we swap the inputs data and multiply by the upstream gradient 
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad 

        out._backward = _backward 
        return out  
    
    def __pow__(self, other): 
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out 
    
    def tanh(self):
        x = self.data 
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), "tanh")

        def _backward(): 
            self.grad += (1 - t**2) * out.grad # The local gradient 1 - t**2 has to be multiplied by the upstream gradient out.grad because of the chain rule 

        out._backward = _backward 
        return out 
    
    def __rmul__(self, other):
        return self * other 
    
    def __truediv__(self, other): 
        return self * other**-1
    
    def __neg__(self):
        return self * -1 
    
    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other 
    
    def exp(self):
        x = self.data 
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        
        out._backward = _backward

        return out 
    
    def backward(self):
        # Topological sort 
        topo = [] 
        visited = set() 
        def build_topo(v):
            if v not in visited: 
                visited.add(v)
                for child in v._prev: 
                    build_topo(child)
                topo.append(v) 
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [45]:
import torch 

x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True 
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True 
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True 
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True 
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print("----")
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
----
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [46]:
torch.Tensor([2.0]).double().dtype

torch.float64

In [ ]:
# creating a neuron based on same principles pytorch used in designing its neural network modules 

from typing import List 

class Neuron: 
    def __init__(self, nin: int):  # nin : number of inputs to the neuron 
        # creates a weight (a number between 1 and -1) for every one of those inputs 
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # use the zip function to pair each input to a weight correspondingly 
        # print(list(zip(self.w, x)))

        # python doesn't output error if nin and input x doesn match so the assert line handles it 
        assert len(x) == len(self.w), f"Expected {len(self.w)} inputs, got {len(x)}"

        # Performs the function w * x + b where w*x is the dot product 
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b) # sum takes a second param which by default is 0.0 but we use self.b here
        out = act.tanh()
        return out 

class Layer: 
    def __init__(self, nin: int, nout: int):
        # create nout number of neurons (using the Neuron class above) with nin number of inputs 
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        # takes the input x and feeds it to every single neuron in the layer (list of neurons created above)
        outs = [n(x) for n in self.neurons]
        return outs 
    
class MLP:
    def __init__(self, nin: int, nouts: List[int]):
        sz = [nin] + nouts # adds list to list [nin, nouts], if nin is 2, nouts = [2, 4, 1] ==> [2, 3, 4, 1]
        # nin: number of inputs to a Neuron obj 
        # nouts: number of Neuron in each layer 

        # sz[i] = nin, sz[i+1] = nouts 
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x 

x = [2.0, 3.0]
n = Neuron(2)
n(x)

Value(data=0.9228144048310731)

In [75]:
a = [2] + [1,3,4]
a 

[2, 1, 3, 4]

In [69]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot